# 50 · Sample a raster over areas, checked against rasterio

## What this checks

**Sample a raster at points or over areas** can total a raster's cells inside each area. This check covers release
0.6.1. It also covers the window of the raster that **Raster input** keeps, the part of the file around the areas.

Here the rasters are three population models for 2025: WorldPop, GRID3 and GHS-POP. Each gives an estimated number
of people per cell. Each is totalled over the 30 census districts of Rwanda and the 28 of Botswana. Each district's
share of its model's total is set beside its share in the 2022 census.

Fieldwork's own test compared those shares rounded to 0.1 %. Here every district's total and cell count is
recomputed, without rounding, from the same files. A wrong total gives a district the wrong denominator for every
rate worked out from it.

## How it is checked

**rasterio**, the Python link to GDAL, reads each raster whole. GDAL is a widely used library for geographic data.
GDAL's own tool marks each district's cells: a cell is in a district when its centre is. **NumPy**, Python's library
for arrays of numbers, adds them up. The district boundaries and the census shares are read from their own files.
No code is shared with Fieldwork.

**The rasters are not in this lab**, nor in the example: they are tens of megabytes each. The lab keeps each file's
address, size and digest. A digest is a fingerprint of a file's exact contents. The check reads the files from a
folder of downloads, named when it is run. It checks each digest before it reads a cell.

Without that folder, the check says so and does not run. This check runs in CPython only: the browser kernel cannot
hold rasters tens of megabytes each, which are not redistributed. In the browser, the notebook says so and skips the
rest.

## What should happen

1. **Cells.** For each district and each model, the same count of cells with a value.
2. **Totals.** Each district's total within one part in a billion (10⁻⁹) of rasterio's. Both sides add the same cells
   in a different order, so tiny differences are expected.
3. **Shares.** Each district's share of its model's total within 10⁻¹². The census shares too, recomputed from the
   census file.

These criteria were fixed before the check was run.

In [ ]:
NEEDED = ["numpy", "rasterio"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
try:
    from checks.check_50_population_districts import run
except ImportError as missing:
    report = None
    print("Not runnable here:", missing)
else:
    report = run(Path.cwd() / "fixtures" / "population-districts-001.json")
    print("agrees:", report["agrees"], report.get("notRun", ""))
    if report["agrees"] is not None:
        c = report["checked"]
        print(f"{c['districts']} districts, {c['totals']} totals, {c['shares']} shares")
        print(f"largest relative gap in a total: {report['largest']['relativeTotal']:.1e}; in a share: {report['largest']['share']:.1e}")
        print("differs:", report["differs"] or "nowhere")

## The result on record

Recorded on 13 November 2026, against release 0.6.1: **Agrees**. It was not rerun with the other checks on
26 November 2026. The rasters on this computer were not in the folder layout the export needs.

All 174 cell counts are equal: 58 districts, three models each. Every total is within 8 × 10⁻¹⁴ of rasterio's,
relative. All 232 shares, the census's included, are within 4 × 10⁻¹⁵.

The result agrees in CPython.

Running the cell above repeats the check on the same data, where the downloads are on the computer.

## What this does not show

- That any model is right. Both sides total the same cells.
- That a district's cells are its people. A cell is counted whole, by where its centre falls, not by the part of it
  inside the district.
- Which district owns a cell whose centre falls exactly on a shared border. Fieldwork counts it in both districts,
  and GDAL in one. Any difference is reported, district by district.
- That GHS-POP's window, cut here from the global file, is the window Fieldwork reads from it. A Fieldwork browser
  test reads that window and gives the same shares to 0.1 %.

In [ ]:
if report is None:
    print("Skipped: this check is not runnable in this kernel (see above).")
else:
    for item in report["notEstablished"]:
        print("-", item)
    print()
    print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
    e = report.get("environment")
    if e:
        print("rasterio:", e["rasterio"], "· GDAL", e["gdal"], "· NumPy", e["numpy"])
        print("runtime :", e["runtime"])